# Notebook 00 — Extract & Save Layer Embeddings (CLS + Patches + BBoxes)

For a configurable number of ImageNet val images and a set of target ViT layers:

1. Load the CLIP ViT model (frozen, eval mode) via `VisionModelWrapper`
2. **Pre-filter** the val set to only images that have bounding box XML annotations
3. Register **custom hooks** capturing the full token sequence `[B, seq_len, d_model]`
   — CLS (position 0) and patch tokens (positions 1:) in one pass
4. Process images in mini-batches
5. Assemble a **list of per-sample bundles** and save to a single `.pt` file

Every sample in the output is guaranteed to have bounding boxes.

---

## Where to get ImageNet bounding box annotations

The bbox annotations are a **separate download** from the images — part of the ILSVRC 2012 devkit.

### Official source (requires free registration)
- **https://image-net.org/download.php** → ILSVRC 2012 → download:
  - `ILSVRC2012_bbox_val_v3.tgz` (~2.3 MB) — val set bounding boxes

### Alternative: Kaggle
`kaggle competitions download -c imagenet-object-localization-challenge`

### Extract
```bash
mkdir -p /opt/models/datasets/imagenet/boxes/val
tar -xzf ILSVRC2012_bbox_val_v3.tgz -C /opt/models/datasets/imagenet/boxes/val
```
Results in a flat directory: `ILSVRC2012_val_00000001.xml` … `ILSVRC2012_val_00050000.xml`
(~20 000 of the 50 000 val images have annotations).

---

## Output `.pt` file schema
```python
{
    'samples': [                          # list of N dicts, one per image
        {
            'val_idx':    int,
            'label':      int,
            'image':      Tensor[3, 224, 224],          # uint8 RGB
            'orig_size':  (W, H),                       # original image dimensions
            'bboxes_orig': [{'xmin','ymin','xmax','ymax','synset'}, ...],  # original px
            'bboxes_224':  [{'xmin','ymin','xmax','ymax','synset'}, ...],  # 224px space
            'logits':     Tensor[num_classes],           # float16
            'cls':        {layer_idx: Tensor[d_model]},  # float16
            'patches':    {layer_idx: Tensor[H, W, d_model]},  # float16
        },
        ...
    ],
    'metadata': dict,
}
```

In [ ]:
%matplotlib inline

import sys
import random
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import torch
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset
from PIL import Image
import torchvision.transforms.functional as TF

sys.path.insert(0, str(Path('..').resolve() / 'src'))

from tuned_lens.model import VisionModelWrapper
from tuned_lens.config import ModelConfig

print('Imports OK')

## Configuration — edit paths and parameters here

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
IMAGENET_VAL_DIR = '/opt/models/datasets/imagenet/extracted/val'
# Flat directory of XMLs from ILSVRC2012_bbox_val_v3.tgz
BBOX_VAL_DIR     = '/opt/models/datasets/imagenet/boxes/val'

# ── Model selection — uncomment the model you want to use ──────────────────
MODEL_NAME = 'vit_large_patch14_clip_224.openai_ft_in1k'   # CLIP ViT-L/14
# MODEL_NAME = 'vit_large_patch16_224.augreg_in21k_ft_in1k'  # ViT-L/16 (AugReg)
# MODEL_NAME = 'deit3_large_patch16_224.fb_in1k'             # DeiT3-L/16
# MODEL_NAME = 'vit_large_patch14_dinov2.lvd142m'           # DINOv2-L/14    (518×518, 37×37 grid)
_MODEL_SHORT = MODEL_NAME.split('.')[0]  # used to namespace output paths

TARGET_LAYERS = [12, 19]   # all three models have 24 blocks (indices 0–23)
N_SAMPLES     = 100                   # number of val images to embed
BATCH_SIZE    = 16                    # images processed per forward pass
SEED          = 42
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR    = Path(f'../outputs/{_MODEL_SHORT}/embeddings')
# ──────────────────────────────────────────────────────────────────────────────

random.seed(SEED)
torch.manual_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / f'embeddings_{MODEL_NAME}_n{N_SAMPLES}.pt'

print(f'Device       : {DEVICE}')
print(f'N_SAMPLES    : {N_SAMPLES}')
print(f'TARGET_LAYERS: {TARGET_LAYERS}')
print(f'Output file  : {output_path}')

## SC 1 — Verify input paths

In [ ]:
assert Path(IMAGENET_VAL_DIR).exists(), f'Val dir not found: {IMAGENET_VAL_DIR}'
assert Path(BBOX_VAL_DIR).exists(),     f'BBox dir not found: {BBOX_VAL_DIR}'

n_xmls = len(list(Path(BBOX_VAL_DIR).glob('*.xml')))
print(f'SC 1 ✓')
print(f'  Val dir  : {IMAGENET_VAL_DIR}')
print(f'  BBox dir : {BBOX_VAL_DIR}  ({n_xmls:,} XML files)')

## Build bbox lookup and define helper functions

### Coordinate transformation
The timm CLIP ViT-L validation transform is:
1. **Resize** so the shortest edge = 224 (bicubic)
2. **Center crop** 224×224

We apply the same steps analytically to each bbox so the coordinates align with the
saved 224×224 images.

In [ ]:
# ── Build stem → xml path mapping ─────────────────────────────────────────────
bbox_dir = Path(BBOX_VAL_DIR)
stem_to_xml: dict[str, Path] = {p.stem: p for p in bbox_dir.glob('*.xml')}
print(f'Indexed {len(stem_to_xml):,} bbox XML files')


# ── Parse a single XML → list of annotation dicts ────────────────────────────
def parse_bbox_xml(xml_path: Path) -> tuple[int, int, list[dict]]:
    """Parse a Pascal-VOC XML and return (orig_w, orig_h, list_of_bboxes).

    Each bbox dict has keys: xmin, ymin, xmax, ymax, synset.
    Coordinates are integers in the original image pixel space.
    """
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    orig_w = int(size.find('width').text)
    orig_h = int(size.find('height').text)
    boxes = []
    for obj in root.findall('object'):
        bb = obj.find('bndbox')
        boxes.append({
            'xmin':   int(float(bb.find('xmin').text)),
            'ymin':   int(float(bb.find('ymin').text)),
            'xmax':   int(float(bb.find('xmax').text)),
            'ymax':   int(float(bb.find('ymax').text)),
            'synset': obj.find('name').text.strip(),
        })
    return orig_w, orig_h, boxes


# ── Transform bbox coords from original → 224×224 space ──────────────────────
def transform_bbox_to_224(
    bbox: dict,
    orig_w: int,
    orig_h: int,
    crop_size: int = 224,
) -> dict | None:
    """Apply timm's resize-shortest-edge + center-crop analytically.

    Returns a new bbox dict with float coords clamped to [0, crop_size],
    or None if the bbox falls entirely outside the crop region.
    """
    # Step 1 — resize so shortest edge = crop_size
    scale = crop_size / min(orig_w, orig_h)
    new_w = orig_w * scale
    new_h = orig_h * scale

    x1 = bbox['xmin'] * scale
    y1 = bbox['ymin'] * scale
    x2 = bbox['xmax'] * scale
    y2 = bbox['ymax'] * scale

    # Step 2 — subtract center-crop offset, then clamp both ends to [0, crop_size]
    left = (new_w - crop_size) / 2
    top  = (new_h - crop_size) / 2

    x1 = max(0.0, min(float(crop_size), x1 - left))
    y1 = max(0.0, min(float(crop_size), y1 - top))
    x2 = max(0.0, min(float(crop_size), x2 - left))
    y2 = max(0.0, min(float(crop_size), y2 - top))

    # Discard boxes that are fully outside the crop (zero area after clamping)
    if x2 <= x1 or y2 <= y1:
        return None

    return {
        'xmin':   round(x1, 2),
        'ymin':   round(y1, 2),
        'xmax':   round(x2, 2),
        'ymax':   round(y2, 2),
        'synset': bbox['synset'],
    }


def transform_bboxes_to_224(boxes: list[dict], orig_w: int, orig_h: int) -> list[dict]:
    """Transform a list of bboxes, dropping any that fall outside the crop."""
    result = []
    for b in boxes:
        transformed = transform_bbox_to_224(b, orig_w, orig_h)
        if transformed is not None:
            result.append(transformed)
    return result


print('Bbox helpers defined.')

## SC 2 — Smoke-test XML parsing

In [ ]:
sample_xml = next(iter(stem_to_xml.values()))
ow, oh, boxes = parse_bbox_xml(sample_xml)

print(f'Test XML      : {sample_xml.name}')
print(f'Original size : {ow} x {oh}')
print(f'Raw bboxes    : {boxes}')

boxes_224 = transform_bboxes_to_224(boxes, ow, oh)
print(f'Transformed   : {boxes_224}  ({len(boxes) - len(boxes_224)} dropped outside crop)')

for b in boxes_224:
    assert 0 <= b['xmin'] <= 224 and 0 < b['xmax'] <= 224, f"xmin/xmax out of range: {b}"
    assert 0 <= b['ymin'] <= 224 and 0 < b['ymax'] <= 224, f"ymin/ymax out of range: {b}"
    assert b['xmin'] < b['xmax'] and b['ymin'] < b['ymax'],  f"degenerate box: {b}"
print('SC 2 ✓')

## Load model

In [ ]:
model_cfg = ModelConfig(
    model_name=MODEL_NAME,
    pretrained=True,
    target_layers=TARGET_LAYERS,
    freeze_model=True,
    patch_mode=False,  # we register our own hooks below
)
wrapper = VisionModelWrapper(model_cfg, device=DEVICE)

H, W = wrapper.patch_grid_size
d    = wrapper.d_model
C    = wrapper.num_classes

print(f'Model       : {MODEL_NAME}')
print(f'd_model     : {d}')
print(f'num_classes : {C}')
print(f'num_layers  : {wrapper.num_layers}')
print(f'patch_grid  : {H}x{W} = {H*W} patches/image')

## Register full-sequence hooks

Remove the wrapper's built-in CLS-only hooks, then attach hooks that capture the
**entire** token sequence so we can slice CLS and patches in one forward pass.

In [ ]:
wrapper._remove_hooks()

_full_states: dict[int, torch.Tensor] = {}
_hook_handles = []

def _make_full_hook(layer_idx: int):
    def hook_fn(module, input, output: torch.Tensor) -> None:
        _full_states[layer_idx] = output.detach()   # [B, 1+H*W, d_model]
    return hook_fn

for layer_idx in TARGET_LAYERS:
    handle = wrapper.model.blocks[layer_idx].register_forward_hook(
        _make_full_hook(layer_idx)
    )
    _hook_handles.append(handle)

print(f'Registered {len(_hook_handles)} full-sequence hooks on layers {TARGET_LAYERS}')

## SC 3 — Smoke-test hooks with a dummy batch

In [ ]:
dummy = torch.zeros(2, 3, 224, 224, device=DEVICE)
with torch.no_grad():
    wrapper.model(dummy)

seq_len_expected = 1 + H * W
for layer_idx in TARGET_LAYERS:
    seq = _full_states[layer_idx]
    assert seq.shape == (2, seq_len_expected, d), (
        f'Layer {layer_idx}: expected (2, {seq_len_expected}, {d}), got {seq.shape}'
    )
    print(f'  Layer {layer_idx:2d}: {tuple(seq.shape)}  ✓')

_full_states.clear()
print(f'SC 3 ✓  seq_len = 1 (CLS) + {H*W} (patches) = {seq_len_expected}')

## Build dataset and sample indices

In [ ]:
transform   = wrapper.get_transform()
val_dataset = datasets.ImageFolder(IMAGENET_VAL_DIR, transform=transform)
raw_dataset = datasets.ImageFolder(IMAGENET_VAL_DIR)   # no transform — for PIL loading
class_names = val_dataset.classes

# ── Pre-filter: keep only val indices that have a bbox XML ────────────────────
annotated_indices = [
    i for i, (img_path, _) in enumerate(val_dataset.imgs)
    if Path(img_path).stem in stem_to_xml
]
print(f'Val images total     : {len(val_dataset):,}')
print(f'With bbox annotation : {len(annotated_indices):,}')

assert N_SAMPLES <= len(annotated_indices), (
    f'N_SAMPLES={N_SAMPLES} exceeds annotated pool size ({len(annotated_indices)})'
)

rng            = random.Random(SEED)
sample_indices = rng.sample(annotated_indices, N_SAMPLES)
subset         = Subset(val_dataset, sample_indices)
loader         = DataLoader(
    subset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=(DEVICE == 'cuda'),
)

print(f'Sampled              : {N_SAMPLES}  (all have bboxes)')
print(f'Batches              : {len(loader)}  (batch_size={BATCH_SIZE})')

## Extract embeddings (batched) and raw images + bboxes (per-sample)

In [ ]:
# Per-layer accumulators (batched GPU extraction)
_acc_cls:     dict[int, list[torch.Tensor]] = {l: [] for l in TARGET_LAYERS}
_acc_patches: dict[int, list[torch.Tensor]] = {l: [] for l in TARGET_LAYERS}
_acc_logits:  list[torch.Tensor] = []
_acc_labels:  list[torch.Tensor] = []

wrapper.model.eval()

for batch_no, (images, labels) in enumerate(loader):
    images = images.to(DEVICE)
    _full_states.clear()
    with torch.no_grad():
        logits = wrapper._run_model(images)   # always returns [B, num_classes] logits
    for layer_idx in TARGET_LAYERS:
        seq = _full_states[layer_idx]                        # [B, 1+H*W, d]
        _acc_cls[layer_idx].append(seq[:, 0, :].cpu())
        _acc_patches[layer_idx].append(seq[:, 1:, :].reshape(-1, H, W, d).cpu())
    _acc_logits.append(logits.cpu())
    _acc_labels.append(labels)
    n_done = min((batch_no + 1) * BATCH_SIZE, N_SAMPLES)
    print(f'  Batch {batch_no+1:3d}/{len(loader)}  ({n_done}/{N_SAMPLES})', end='\r')

# Concatenate into full tensors, cast embeddings to float16
cls_all     = {l: torch.cat(_acc_cls[l],     dim=0).half() for l in TARGET_LAYERS}  # [N, d]
patches_all = {l: torch.cat(_acc_patches[l], dim=0).half() for l in TARGET_LAYERS}  # [N, H, W, d]
logits_all  = torch.cat(_acc_logits, dim=0).half()   # [N, C]
labels_all  = torch.cat(_acc_labels, dim=0)          # [N]  int64
print(f'\nEmbedding extraction done.')

# ── Per-sample: load raw image + parse bboxes ─────────────────────────────────
print('Loading raw images and bounding boxes...')
n_dropped_boxes = 0
samples = []
for i, val_idx in enumerate(sample_indices):
    img_path = Path(raw_dataset.imgs[val_idx][0])
    stem     = img_path.stem

    pil_img        = Image.open(img_path).convert('RGB')
    orig_w, orig_h = pil_img.size
    img_224        = pil_img.resize((224, 224), Image.BILINEAR)
    image_t        = torch.from_numpy(np.array(img_224)).permute(2, 0, 1)  # [3, 224, 224] uint8

    xml_path           = stem_to_xml[stem]   # guaranteed — pre-filtered
    ow, oh, boxes_orig = parse_bbox_xml(xml_path)
    boxes_224          = transform_bboxes_to_224(boxes_orig, ow, oh)
    n_dropped_boxes   += len(boxes_orig) - len(boxes_224)

    samples.append({
        'val_idx':     val_idx,
        'label':       int(labels_all[i].item()),
        'image':       image_t,                                      # uint8 [3, 224, 224]
        'orig_size':   (orig_w, orig_h),
        'bboxes_orig': boxes_orig,
        'bboxes_224':  boxes_224,
        'logits':      logits_all[i],                                # float16 [C]
        'cls':     {l: cls_all[l][i]     for l in TARGET_LAYERS},   # float16 [d]
        'patches': {l: patches_all[l][i] for l in TARGET_LAYERS},   # float16 [H, W, d]
    })

    if (i + 1) % 20 == 0 or (i + 1) == N_SAMPLES:
        print(f'  {i+1}/{N_SAMPLES} samples assembled', end='\r')

print(f'\nDone. {len(samples)} bundles assembled.')
print(f'Boxes dropped (fully outside crop): {n_dropped_boxes}')

## SC 4 — Verify every bundle is complete and well-formed

In [ ]:
assert len(samples) == N_SAMPLES, f'Expected {N_SAMPLES} bundles, got {len(samples)}'

required_keys = {'val_idx', 'label', 'image', 'orig_size',
                 'bboxes_orig', 'bboxes_224', 'logits', 'cls', 'patches'}

for i, s in enumerate(samples):
    assert set(s.keys()) == required_keys, f'Sample {i} missing keys: {required_keys - set(s.keys())}'

    assert s['image'].shape == (3, 224, 224),      f'Sample {i} image shape: {s["image"].shape}'
    assert s['image'].dtype == torch.uint8,         f'Sample {i} image dtype: {s["image"].dtype}'
    assert s['logits'].shape == (C,),               f'Sample {i} logits shape: {s["logits"].shape}'
    assert s['logits'].dtype == torch.float16
    assert len(s['bboxes_orig']) > 0,               f'Sample {i} has no bboxes (should have been filtered)'
    # bboxes_224 may be shorter than bboxes_orig — boxes outside the crop are dropped
    assert len(s['bboxes_224']) <= len(s['bboxes_orig'])

    for l in TARGET_LAYERS:
        assert s['cls'][l].shape    == (d,),         f'Sample {i} layer {l} CLS shape wrong'
        assert s['patches'][l].shape == (H, W, d),   f'Sample {i} layer {l} patches shape wrong'
        assert s['cls'][l].dtype    == torch.float16
        assert s['patches'][l].dtype == torch.float16

    for b in s['bboxes_224']:
        assert 0 <= b['xmin'] <= 224 and 0 <= b['xmax'] <= 224
        assert 0 <= b['ymin'] <= 224 and 0 <= b['ymax'] <= 224

# Summary stats
n_boxes_total = sum(len(s['bboxes_orig']) for s in samples)
n_boxes_224   = sum(len(s['bboxes_224'])  for s in samples)
print(f'Bundles       : {len(samples)}')
print(f'Total bboxes  : {n_boxes_total} orig  →  {n_boxes_224} after crop  ({n_boxes_total - n_boxes_224} dropped)')
print(f'Label range   : [{min(s["label"] for s in samples)}, {max(s["label"] for s in samples)}]')
preds    = torch.stack([s['logits'] for s in samples]).float().argmax(dim=1)
gt       = torch.tensor([s['label'] for s in samples])
top1_acc = (preds == gt).float().mean().item()
print(f'Top-1 acc     : {top1_acc:.3%}')
print('SC 4 ✓')

## Save to disk

In [ ]:
payload = {
    'samples': samples,   # List[dict] — N self-contained bundles, each with:
                          #   val_idx, label, image, orig_size,
                          #   bboxes_orig, bboxes_224,
                          #   logits, cls, patches
    'metadata': {
        'model_name':    MODEL_NAME,
        'n_samples':     N_SAMPLES,
        'target_layers': TARGET_LAYERS,
        'patch_grid':    (H, W),
        'd_model':       d,
        'num_classes':   C,
        'class_names':   class_names,        # List[str] — 1000 ImageNet synset IDs in label order
        'imagenet_val_dir': IMAGENET_VAL_DIR,
        'bbox_val_dir':     BBOX_VAL_DIR,
        'seed':          SEED,
        'embedding_dtype': 'float16',
        'image_dtype':     'uint8',
        'bbox_note':       'all samples have bboxes; bboxes_224 coords are in [0,224]',
        'bbox_transform':  'resize_shortest_edge_224_then_center_crop_224',
    },
}

torch.save(payload, output_path)

size_mb = output_path.stat().st_size / (1024 ** 2)
print(f'Saved → {output_path}')
print(f'File size: {size_mb:.1f} MB')

## Reload and verify (round-trip check)

In [ ]:
loaded = torch.load(output_path, map_location='cpu', weights_only=False)

assert set(loaded.keys()) == {'samples', 'metadata'}
assert len(loaded['samples']) == N_SAMPLES

s0 = loaded['samples'][0]
assert s0['image'].shape  == (3, 224, 224)
assert s0['image'].dtype  == torch.uint8
assert s0['logits'].shape == (C,)
assert s0['logits'].dtype == torch.float16
assert len(s0['bboxes_orig']) > 0

for l in TARGET_LAYERS:
    assert s0['cls'][l].shape     == (d,)
    assert s0['patches'][l].shape == (H, W, d)

print('Round-trip reload ✓')
print(f"Metadata : {loaded['metadata']}")
print(f"\nSample 0 keys  : {list(s0.keys())}")
print(f"  label        : {s0['label']}  ({class_names[s0['label']]})")
print(f"  orig_size    : {s0['orig_size']}")
print(f"  bboxes_orig  : {s0['bboxes_orig']}")
print(f"  bboxes_224   : {s0['bboxes_224']}")

## Visualization — 5 samples: image + bboxes | per-layer patch L2-norm heatmap

In [ ]:
import matplotlib.patches as mpatches

# ── Pull everything from the loaded file only ──────────────────────────────────
_meta        = loaded['metadata']
_class_names = _meta['class_names']
_layers      = _meta['target_layers']
_H, _W       = _meta['patch_grid']

N_VIZ       = 5
n_cols      = 1 + len(_layers)    # image+bbox | one heatmap per layer

rng_viz     = random.Random(_meta['seed'] + 1)
viz_samples = rng_viz.sample(loaded['samples'], N_VIZ)

for idx, s in enumerate(viz_samples):
    fig, axes = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 3.8))

    # ── Col 0: image with bounding boxes ──────────────────────────────────────
    ax_img = axes[0]
    img_np = s['image'].permute(1, 2, 0).numpy()   # [224, 224, 3]  uint8
    ax_img.imshow(img_np)

    colors = plt.cm.tab10.colors
    for bi, bbox in enumerate(s['bboxes_224']):
        x, y   = bbox['xmin'], bbox['ymin']
        bw, bh = bbox['xmax'] - x, bbox['ymax'] - y
        color  = colors[bi % len(colors)]
        ax_img.add_patch(mpatches.Rectangle(
            (x, y), bw, bh,
            linewidth=1.5, edgecolor=color, facecolor='none',
        ))
        ax_img.text(
            x, max(y - 3, 0), bbox['synset'],
            fontsize=5.5, color=color,
            bbox=dict(facecolor='black', alpha=0.45, pad=1, linewidth=0),
        )

    ax_img.set_title(_class_names[s['label']], fontsize=7)
    ax_img.axis('off')

    # ── Cols 1…N: per-layer patch L2-norm heatmap ─────────────────────────────
    for col, layer_idx in enumerate(_layers, start=1):
        ax        = axes[col]
        norm_map  = s['patches'][layer_idx].float().norm(dim=-1).numpy()  # [H, W]
        im        = ax.imshow(norm_map, cmap='viridis', interpolation='nearest')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(f'patch ‖·‖₂\nlayer {layer_idx}', fontsize=7)
        ax.axis('off')

    fig.suptitle(
        f'Sample {idx+1}/{N_VIZ}  val_idx={s["val_idx"]}  '
        f'{len(s["bboxes_224"])} bbox(es)',
        fontsize=8, y=1.01,
    )
    plt.tight_layout()
    plt.show()

## Cleanup hooks

In [ ]:
for handle in _hook_handles:
    handle.remove()
_hook_handles.clear()
_full_states.clear()

print('Hooks removed. Done.')